# 8. 공통 차종 선택과 챗봇 UI

수업 `15_streamlit/st_6_chatbot.py`의 `st.session_state`, `st.chat_message`, `st.chat_input`, `st.cache_resource`를 사용합니다.

**공통 화면에서 차종 선택 → 해당 차종의 에이전트와 대화**하는 구조입니다. 현재 실제 연결된 차종은 현대 아이오닉 5 하나입니다. 나머지 세 자리는 준비 중이며, 다른 차종 질문을 아이오닉 5 자료로 대신 답하지 않습니다.

디자인은 공유받은 [Figma UI Kit](https://www.figma.com/design/5JkRd77oqKd2bkODFtO6nR/3-Free-AI-Chatbot-App-UI-Kit--Community-?node-id=10301-21963)의 흰 배경·보라색 포인트·둥근 말풍선을 참고했습니다. 첫 화면은 이후 제공된 이미지에 맞춰 사이드바 없이 인사말과 차종 카드만 두었습니다. 수업 도구인 Streamlit에 맞춘 구현이며 픽셀 단위 복제는 아닙니다.

이 노트북은 실행과 구조를 설명합니다. 실제 웹 UI는 `chatbot/app.py`입니다. 노트북 전체를 웹 서버에서 exec하지 않습니다.

## 1. 파일별 역할

| 파일 | 역할 |
|---|---|
| chatbot/app.py | 차종 카드 선택, 대화 말풍선, 입력창, 출처·그림 표시 |
| chatbot/registry.py | 차종 이름과 전용 모듈의 연결 목록 |
| chatbot/ioniq5_backend.py | 7번의 검색 도구·에이전트·세션/기록 함수를 웹에서 사용할 모듈 |

팀원은 동일한 `start_session`, `end_session`, `chat`, `get_session`, `get_related_images` 함수 규약의 어댑터를 만든 후 registry에 등록할 수 있습니다. 현재 다른 팀원 코드는 수정하거나 연결하지 않았습니다.

모델/검색기는 재사용하지만 대화 ID와 화면 메시지는 사용자 연결마다 별도로 관리합니다. 차종을 바꾸면 기존 세션을 닫고 새 차종의 새 세션을 시작합니다.

In [ ]:
from pathlib import Path
import subprocess, socket
from IPython.display import display, Markdown

project_root = Path('/Users/anna/mle-02-p1-team1')
python_path = project_root / '.venv/bin/python'
app_path = project_root / 'src/car_search_rag/anna_rag/chatbot/app.py'
assert python_path.exists() and app_path.exists()
print('실행 파일:', app_path)

## 2. 설치와 실행

Streamlit 의존성은 pyproject.toml과 uv.lock에 반영했습니다. 다른 PC에서는 프로젝트 루트에서 `uv sync`를 먼저 실행하세요. 접속 정보는 기존 `.env`를 서버에서만 읽습니다.

아래 셀은 로컬 서버를 시작합니다. 8501 포트가 이미 사용 중이면 다른 서버를 종료하지 않고 안내합니다. 같은 노트북에서 만든 프로세스만 뒤의 종료 셀로 중지합니다. 공개 배포는 하지 않으며 같은 컴퓨터의 브라우저에서 접속합니다.

In [ ]:
port = 8501
url = f'http://127.0.0.1:{port}'
with socket.socket() as sock:
    in_use = sock.connect_ex(('127.0.0.1', port)) == 0
if in_use:
    print('8501 포트가 사용 중입니다. 아래 주소가 이 챗봇인지 확인하세요. 다른 앱이면 port 값을 바꾸세요.')
else:
    server = subprocess.Popen(
        [str(python_path), '-m', 'streamlit', 'run', str(app_path),
         '--server.address', '127.0.0.1', '--server.port', str(port),
         '--server.headless', 'true', '--browser.gatherUsageStats', 'false',
         '--theme.base', 'light', '--theme.primaryColor', '#6350db'],
        cwd=project_root
    )
display(Markdown(f'[챗봇 열기]({url})'))

## 3. 화면에서 시험하기

1. 첫 화면에는 인사말과 차량 카드 4개, 비활성 입력창이 표시됩니다.
2. 아이오닉 5 카드를 누르면 바로 대화를 시작합니다. 산타페·소나타·제네시스는 연결 전까지 준비 중입니다.
3. “충전 커넥터가 안 빠져요”처럼 질문합니다.
4. 같은 화면에서 “그 케이블은 어디 있어?”처럼 이어 질문합니다.
5. 출처 펼치기와 관련 그림을 확인합니다.
6. 대화 종료를 누르고 차종 선택 화면에서 다시 입장하면 화면 대화와 맥락이 새로 시작됩니다.

실제 질문은 API 비용이 발생하고 DB에 기록됩니다. 현재 UI는 실습 기록(is_test=True)으로 저장합니다. 대시보드에서 테스트 기록 필터로 조회할 수 있습니다.

## 4. 수업 코드와 연결해서 이해하기

- `st.session_state`: 현재 브라우저 연결의 차종·대화 ID·말풍선을 보관합니다.
- `st.cache_resource`: 모델과 검색기 준비를 매번 반복하지 않습니다. 메시지 내용은 공유 캐시에 넣지 않습니다.
- `st.chat_input`: 질문을 입력받습니다.
- `backend.chat`: 해당 차종의 RAG 에이전트와 기록 저장을 호출합니다.
- `st.chat_message`: 질문과 답변을 말풍선으로 보여줍니다.
- `st.expander`: 출처와 그림을 필요할 때 펼쳐 볼 수 있게 합니다.

기존 DB 연결 클래스의 공유 connection 속성에 의존하지 않도록 백엔드에서 호출마다 새 DatabaseManager를 사용합니다. 웹의 여러 대화가 한 DB 연결을 잘못 공유하지 않게 하기 위한 조정입니다.

## 5. 이탈과 기록의 정확한 동작

- **대화 종료 후 차종 변경:** DB 세션을 종료하고 화면 맥락을 지웁니다.
- **새로고침·재입장:** Streamlit의 새 브라우저 연결에서는 새로운 세션을 시작합니다. URL이나 로컬 저장소에서 이전 대화 ID를 복원하지 않습니다.
- **탭 닫힘:** 닫힘 신호로 DB 종료 시각을 즉시 확정하지는 않습니다. 이전 ID는 새 화면에서 재사용하지 않습니다.
- **30분 무활동:** 다음 입력 시 만료를 확인하고 차종 선택으로 돌아갑니다. 예약 정리 작업은 아직 없습니다.

현재는 로컬 실습 UI입니다. 외부 서비스 공개 전에 인증/세션 소유권 검증, 개인정보 보관 정책, 강제 종료로 남은 running 요청의 복구를 추가해야 합니다. API 키나 DB_URL을 화면에 넣지 마세요.

## 6. 직접 시작한 서버 종료 (선택)

아래 코드는 이 노트북의 실행 셀에서 만든 프로세스만 종료합니다. 다른 곳에서 실행한 서버는 해당 터미널에서 Ctrl+C로 종료하세요.

In [ ]:
STOP_SERVER = False
if STOP_SERVER:
    if 'server' in globals() and server.poll() is None:
        server.terminate()
        server.wait(timeout=10)
        print('이 노트북에서 시작한 서버를 종료했습니다.')
    else:
        print('이 노트북이 실행 중인 서버가 없습니다.')

## 다음 단계

채팅 UI를 먼저 사용해 보고, 이후 데이터·도구 가이드와 품질 대시보드를 완성합니다. 먼저 생성됐던 대시보드는 `9_품질_대시보드_초안.ipynb`로 이동해 보류했습니다. 다른 차종의 실제 에이전트 통합은 현재 작업 범위에서 제외합니다.

## 구현 검증
Streamlit AppTest로 차종 미선택/준비 중 차종 차단, 아이오닉 5 입장, 질문·후속 질문, 출처 출력, 종료·재입장 시 새 대화 ID와 빈 메시지 목록을 확인했습니다. 브라우저에서도 실제 RAG 답변과 PDF 출처 표시를 확인했습니다. UI 실습 질문은 is_test=True로 DB에 저장됩니다.
디자인은 참고 Figma의 분위기를 Streamlit에서 적용한 것입니다. 웹브라우저를 닫았다고 DB의 종료 시각이 즉시 기록되는 것은 아니며, 재입장 시 이전 세션을 복원하지 않는 방식으로 사용자 맥락을 분리합니다.